# **🤗 Transformer Pipeline Inference**

## **What's Covered?**
1. Pipelines
    - What is pipeline()?
    - Behind the Scenes
    - Key Points
2. Pipeline Syntax 
    - Example Usage
    - Garbage Collection
3. Default Model List
    - Common Tasks Supported
    - Identify the Pipeline Supported Tasks

## **Pipelines**

### **What is pipeline()?**
The `pipeline()` makes it simple to use any model from the `Hub` for inference on any language, computer vision, speech, and multimodal tasks. Even if you don’t have experience with a specific modality or aren’t familiar with the underlying code behind the models, you can still use them for inference with the `pipeline()`! 

It is the most powerful way to start using pre-trained Hugging Face models. 

It's a high level API that abstracts away all the complexity of tokenization, model loading, and post-processing, allowing you to perform common tasks with just a few lines of code.

### **Behind the Scenes**
- Determine framework (py/tf/jax)
- Loads tokenizer
- Loads model
- Choose Device (MPS/CUDA/CPU)
- Handles pre/post-processing
- Gives results

### **Key Points**
- The first time you run a pipeline for a specific model, it will download the model weights (which can be several hundred MB to GBs).
- Subsequent runs will use the cached version.
- You can specify a particular model within the pipeline if you don't want the default.
- The output format of the pipeline varies depending on the task.

## **Pipeline Syntax**
1. Start by importing `pipeline` and `torch`.
```python
from transformers import pipeline
import torch
```
2. Specify the inference task, model and torch_dtype.
    - `torch_dtype` tells the HuggingFace pipeline in which numeric precision the model weights and computations should be loaded.
    - `torch_dtype=torch.bfloat16` means Load the model parameters in `bfloat16` precision instead of the default (usually float32).
    - This is done for Lower Memory Usage and Faster Inference.
```python
classifier = pipeline(
    task="text-classification", 
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    torch_dtype=torch.bfloat16,
)
```
3. Pass the input to the `pipeline()`
```python
classifier(input_text)
```


### **Example Usage**

In [1]:
# Import pipeline
from transformers import pipeline
import torch

# Specify the inference task
classifier = pipeline(
    task="text-classification", 
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    torch_dtype=torch.bfloat16,
)

classifier("It was a very bad movie.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997997879981995}]

In [2]:
classifier.device

device(type='mps', index=0)

**Important**
Transformers needs to decide:
- Should I load a PyTorch model? (pt)
- Should I load a TensorFlow model? (tf)
- Which AutoModel class is correct?
- If the user didn’t specify a model, which model should I default to?

### **Garbage Collection**

```python
del classifier
```
- This does not delete the object from memory directly.
- It only removes the name `classifier` from the current namespace.
- `classifier` is just a variable name. That name was pointing to some object. `del` translator removes that reference.

```python
import gc
gc.collect()
# Ouput: 10
```
- This explicitly asks Python’s garbage collector to find unreachable objects and free them.
- Output represents the number of unreachable objects which were found and collected. 

In [3]:
del classifier

import gc
gc.collect()

261

## **Default Model List**

### **Common Tasks Supported**
- text-classification
- token-classification
- text-generation
- ner
- question-answering
- fill-mask (predicting missing words)
- zero-shot-classification (classifying text without specific training examples)
- ... and many more!

Explore more on:  
https://huggingface.co/docs/transformers/main_classes/pipelines

### **Identify the Pipeline Supported Tasks**

In [4]:
from transformers.pipelines import SUPPORTED_TASKS
print(SUPPORTED_TASKS.keys())

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'image-to-image', 'keypoint-matching', 'any-to-any'])


In [5]:
SUPPORTED_TASKS["audio-classification"]

{'impl': transformers.pipelines.audio_classification.AudioClassificationPipeline,
 'pt': (transformers.models.auto.modeling_auto.AutoModelForAudioClassification,),
 'default': {'model': ('superb/wav2vec2-base-superb-ks', '372e048')},
 'type': 'audio'}

In [6]:
SUPPORTED_TASKS["text-classification"]

{'impl': transformers.pipelines.text_classification.TextClassificationPipeline,
 'pt': (transformers.models.auto.modeling_auto.AutoModelForSequenceClassification,),
 'default': {'model': ('distilbert/distilbert-base-uncased-finetuned-sst-2-english',
   '714eb0f')},
 'type': 'text'}

In [7]:
import pandas as pd

rows = []

for task, info in SUPPORTED_TASKS.items():
    
    # Extract pipeline class name
    impl = info.get("impl")
    impl_name = impl.__name__ if impl else None
    
    # Extract supported PT models
    pt_models = info.get("pt", ())
    pt_model_names = [m.__name__ for m in pt_models]
    
    # Extract default model info
    default = info.get("default", {})
    model_name = None
    revision = None
    
    if "model" in default:
        model = default["model"]
        
        # Some tasks store model differently
        if isinstance(model, tuple):
            model_name, revision = model
        
        elif isinstance(model, dict):
            model_name, revision = model.get("pt", (None, None))
    
    rows.append({
        "task": task,
        "pipeline": impl_name,
        "default_model": model_name,
        "task_type": info.get("type"),
        "pt_models": ", ".join(pt_model_names)
    })

df = pd.DataFrame(rows)

df

,task,pipeline,default_model,task_type,pt_models
0,audio-classification,AudioClassificationPipeline,superb/wav2vec2-base-superb-ks,audio,AutoModelForAudioClassification
1,automatic-speech-recognition,AutomaticSpeechRecognitionPipeline,facebook/wav2vec2-base-960h,multimodal,"AutoModelForCTC, AutoModelForSpeechSeq2Seq"
2,text-to-audio,TextToAudioPipeline,suno/bark-small,text,"AutoModelForTextToWaveform, AutoModelForTextTo..."
3,feature-extraction,FeatureExtractionPipeline,distilbert/distilbert-base-cased,multimodal,AutoModel
4,text-classification,TextClassificationPipeline,distilbert/distilbert-base-uncased-finetuned-s...,text,AutoModelForSequenceClassification
5,token-classification,TokenClassificationPipeline,dbmdz/bert-large-cased-finetuned-conll03-english,text,AutoModelForTokenClassification
6,question-answering,QuestionAnsweringPipeline,distilbert/distilbert-base-cased-distilled-squad,text,AutoModelForQuestionAnswering
7,table-question-answering,TableQuestionAnsweringPipeline,google/tapas-base-finetuned-wtq,text,AutoModelForTableQuestionAnswering
8,visual-question-answering,VisualQuestionAnsweringPipeline,dandelin/vilt-b32-finetuned-vqa,multimodal,AutoModelForVisualQuestionAnswering
9,document-question-answering,DocumentQuestionAnsweringPipeline,impira/layoutlm-document-qa,multimodal,AutoModelForDocumentQuestionAnswering
